# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
# Set default catalog and schema
schema = dbutils.widgets.get("schema")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

In [0]:
from pyspark.sql import functions as f

# NYC Taxi sample report
# samples.nyctaxi.trips covers 2016 data, so scope the date filter to a single
# month to avoid a full-table scan.
trips = (
    spark.read.table("samples.nyctaxi.trips")
    .filter(
        "tpep_pickup_datetime >= '2016-01-01' AND tpep_pickup_datetime < '2016-02-01'"
    )
)

# High-level summary metrics
summary = trips.agg(
    f.count("*").alias("total_trips"),
    f.round(f.avg("trip_distance"), 2).alias("avg_distance_mi"),
    f.round(f.avg("fare_amount"), 2).alias("avg_fare_usd"),
    f.round(f.sum("fare_amount"), 0).alias("total_fares_usd"),
    f.round(f.avg(f.unix_timestamp("tpep_dropoff_datetime") - f.unix_timestamp("tpep_pickup_datetime")) / 60, 1).alias("avg_duration_min"),
)
display(summary)

# Top 10 busiest pickup zones
busiest_zones = (
    trips.groupBy("pickup_zip")
    .agg(
        f.count("*").alias("trips"),
        f.round(f.avg("fare_amount"), 2).alias("avg_fare_usd"),
    )
    .orderBy(f.desc("trips"))
    .limit(10)
)
display(busiest_zones)

# A peek at raw rows
display(
    trips.select(
        "tpep_pickup_datetime",
        "pickup_zip",
        "dropoff_zip",
        "trip_distance",
        "fare_amount",
        (f.unix_timestamp("tpep_dropoff_datetime") - f.unix_timestamp("tpep_pickup_datetime")).alias("trip_seconds"),
    ).limit(5)
)
